# Practical P19: Output Parsers & Structured Output with LangChain & Google Gemini
**Course**: PGDCA — Hands-On Large Language Models  
**Unit 3**: LLM Frameworks for Application Development  
**Syllabus Topic**: 3.1 LangChain framework: chains, memory, output parsers  
**Model Provider**: Google Gemini (`gemini-1.5-flash`) via `langchain-google-genai`  
**Learning Outcome**: Master LangChain output parsers (`StrOutputParser`, `JsonOutputParser`, `PydanticOutputParser`), enforce strict type validation with Pydantic schemas, and reliably extract structured JSON from real Google Gemini responses.

## Part 1: Theoretical Foundations — Why Structured Output Matters

### 1.1 The Challenge with Raw LLM Text
LLMs are probabilistic autoregressive token generators. By default, they return free-form text or Markdown prose. However, software systems require **deterministic, validated data structures**:
* User interfaces require JSON objects to render charts or cards.
* Databases require validated integers, floats, booleans, and strings.
* Backend APIs fail when expected keys are missing or formatted incorrectly.

### 1.2 What is an Output Parser?
An **Output Parser** in LangChain handles three vital responsibilities:
1. **Instruction Injection**: Automatically generates prompt instructions (`parser.get_format_instructions()`) that explain the target schema to Google Gemini.
2. **Text Extraction**: Strips preamble conversational chatter, markdown fences (```` ```json ````), and whitespace.
3. **Validation & Casting**: Parses the JSON string and casts values into strongly typed Python / Pydantic objects.

```mermaid
graph LR
    UserPrompt["Prompt with Format Instructions"] --> Gemini["Google Gemini (gemini-1.5-flash)"]
    Gemini --> RawOutput["Raw String with JSON Payload"]
    RawOutput --> Parser["PydanticOutputParser"]
    Parser --> ValidatedObj["Validated Pydantic Instance"]
```

### 1.3 Core Output Parsers in LangChain:
| Parser | Description | Best Used For |
| :--- | :--- | :--- |
| **`StrOutputParser`** | Extracts text string from message | Plain text answers, summaries |
| **`JsonOutputParser`** | Parses response into standard Python `dict` or `list` | Flexible key-value structures |
| **`PydanticOutputParser`** | Validates response against a typed `BaseModel` schema | Type-safe enterprise schemas |

## Part 2: Defining Type-Safe Schemas with Pydantic

Pydantic is Python's standard data validation library. It uses Python type annotations to validate data boundaries at runtime.

In [1]:
# Part 2 Code: Defining and validating schemas with Pydantic
from pydantic import BaseModel, Field, ValidationError
from typing import List

# Define a strongly typed Course Evaluation schema
class CourseReview(BaseModel):
    course_name: str = Field(description="Title of the course")
    rating: float = Field(ge=1.0, le=5.0, description="Overall rating from 1.0 to 5.0")
    key_topics: List[str] = Field(description="List of main topics covered")
    recommended: bool = Field(description="True if recommended for students, False otherwise")

# 1. Validating conforming data
valid_data = {
    "course_name": "PGDCA - Hands-On LLM",
    "rating": 4.8,
    "key_topics": ["LangChain", "LCEL", "Output Parsers", "Google Gemini"],
    "recommended": True
}
review_obj = CourseReview(**valid_data)
print("✅ Pydantic Object Validated Successfully:")
print(f"Course: {review_obj.course_name} | Rating: {review_obj.rating} | Recommended: {review_obj.recommended}")

# 2. Demonstrating validation error rejection
try:
    invalid_data = {"course_name": "Bad Data", "rating": 99.0, "key_topics": "not-a-list", "recommended": "maybe"}
    CourseReview(**invalid_data)
except ValidationError as e:
    print("\n🛡️ Pydantic Rejection Caught Invalid Data:")
    print(f"Errors detected: {len(e.errors())} field violations")

✅ Pydantic Object Validated Successfully:
Course: PGDCA - Hands-On LLM | Rating: 4.8 | Recommended: True

🛡️ Pydantic Rejection Caught Invalid Data:
Errors detected: 3 field violations


## Part 3: Authentic LangChain `PydanticOutputParser` with Google Gemini

Let's combine `PydanticOutputParser` with a `PromptTemplate` and Google Gemini.
Notice how `parser.get_format_instructions()` automatically generates the exact prompt specification for Gemini!

In [2]:
# Part 3 Code: Real Google Gemini with PydanticOutputParser
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# 1. Instantiate the parser with the target Pydantic schema
parser = PydanticOutputParser(pydantic_object=CourseReview)

# 2. Inspect auto-generated format instructions
print("=" * 60)
print("📝 AUTO-GENERATED FORMAT INSTRUCTIONS (Injected into Gemini prompt):")
print("=" * 60)
print(parser.get_format_instructions())

# 3. Create PromptTemplate with format instructions
prompt = PromptTemplate(
    template="""Analyze the following student feedback and extract the structured course review.
{format_instructions}

Student Feedback:
{feedback}
""",
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. Real Google Gemini Model (temperature=0.0 for deterministic JSON formatting)
gemini_structured = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    api_key=api_key or "AIzaSy_placeholder_until_env_key_set",
    temperature=0.0
)

# 5. Compose LCEL Chain: prompt -> model -> parser
structured_chain = prompt | gemini_structured | parser

# 6. Execute chain on raw unstructured feedback
sample_feedback = "I took the PGDCA Advanced LLMs course and it was phenomenal (rating it 4.9/5). We learned LangChain Pipelines, Structured Output with Gemini, and Prompt Design. Highly recommend it to all computer science students!"

try:
    parsed_review = structured_chain.invoke({"feedback": sample_feedback})
    print("=" * 60)
    print("🎯 PARSED & TYPED PYDANTIC OBJECT FROM REAL GEMINI:")
    print("=" * 60)
    print("Type:", type(parsed_review))
    print("Course Name: ", parsed_review.course_name)
    print("Rating:      ", parsed_review.rating)
    print("Key Topics:  ", parsed_review.key_topics)
    print("Recommended: ", parsed_review.recommended)
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this structured extraction live with Gemini.")
    print(f"Notice: {e}")

📝 AUTO-GENERATED FORMAT INSTRUCTIONS (Injected into Gemini prompt):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"course_name": {"description": "Title of the course", "title": "Course Name", "type": "string"}, "rating": {"description": "Overall rating from 1.0 to 5.0", "maximum": 5.0, "minimum": 1.0, "title": "Rating", "type": "number"}, "key_topics": {"description": "List of main topics covered", "items": {"type": "string"}, "title": "Key Topics", "type": "array"}, "recommended": {"description": "True if recommended for students, False otherwise", "title": "Recommended",

## Part 4: Native Gemini Structured Output (`.with_structured_output()`)

Modern models like Google Gemini support native structured outputs directly at the API level using function calling / JSON schema modes.
LangChain provides `.with_structured_output(Schema)` for seamless schema enforcement!

In [3]:
# Part 4 Code: Native Gemini Structured Output
# Using .with_structured_output() with Google Gemini
native_structured_gemini = gemini_structured.with_structured_output(CourseReview)

try:
    feedback_text = "The PGDCA course on LLMs was great, 4.7 rating. We covered chains and memory. Definitely recommended."
    native_result = native_structured_gemini.invoke(feedback_text)
    print("=" * 60)
    print("⚡ GEMINI NATIVE STRUCTURED OUTPUT RESULT:")
    print("=" * 60)
    print("Result:", native_result)
    print(f"Course: {native_result.course_name} | Rating: {native_result.rating}")
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run native structured outputs live with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to run native structured outputs live with Gemini.
Notice: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


## Part 5: Hands-On Student Exercise

**Objective**: Build a structured "Invoice & Receipt Extractor" using real Google Gemini:
1. Define an `Invoice` Pydantic model with `merchant_name`, `invoice_number`, `items` (list of items with `name` and `price`), and `total_amount`.
2. Configure a `PydanticOutputParser` and compose an LCEL chain with Gemini to extract the structured invoice from raw text.

In [4]:
# Student Exercise: Invoice & Receipt Extractor with Google Gemini
class InvoiceItem(BaseModel):
    item_name: str = Field(description="Name of the purchased item")
    price: float = Field(description="Price of the item in USD")

class Invoice(BaseModel):
    merchant_name: str = Field(description="Name of the store or merchant")
    invoice_number: str = Field(description="Invoice identifier number")
    items: List[InvoiceItem] = Field(description="List of purchased items")
    total_amount: float = Field(description="Total invoice sum in USD")

# 1. Setup Parser and Prompt
invoice_parser = PydanticOutputParser(pydantic_object=Invoice)

invoice_prompt = PromptTemplate(
    template="""Parse the invoice details from the receipt text.
{format_instructions}
Receipt Text:
{receipt}
""",
    input_variables=["receipt"],
    partial_variables={"format_instructions": invoice_parser.get_format_instructions()}
)

receipt_raw_text = """
ACME Tech Supplies
Invoice #INV-2026-9811
Items:
- USB-C Hub: $35.50
- HDMI Cable: $14.50
Total: $50.00
"""

invoice_pipeline = invoice_prompt | gemini_structured | invoice_parser

try:
    extracted_invoice = invoice_pipeline.invoke({"receipt": receipt_raw_text})
    print("=" * 60)
    print("🧾 REAL GEMINI EXTRACTED STRUCTURED INVOICE:")
    print("=" * 60)
    print("Merchant: ", extracted_invoice.merchant_name)
    print("Invoice #: ", extracted_invoice.invoice_number)
    print("Total:     $", extracted_invoice.total_amount)
    print("Line Items:")
    for item in extracted_invoice.items:
        print(f"  * {item.item_name}: ${item.price:.2f}")
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this invoice extraction live with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to run this invoice extraction live with Gemini.
Notice: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
